# CrewAI Customer Support Swarm (Gemini) — Line-by-Line Explanation

This notebook builds a **multi-agent customer support system** using **CrewAI**, with
Google's **Gemini** as the underlying LLM. Three specialised agents collaborate:

- **Triage Agent** — classifies each incoming message as `technical`, `refund`, or
  `general`, and assigns a priority.
- **Technical Support Agent** — searches a mock knowledge base to resolve technical
  issues, or honestly escalates when it can't.
- **Refund Operations Agent** — extracts refund details from the customer's message,
  asks clarifying questions when information is missing, and — critically — never
  makes the actual eligibility/processing decision itself; that's enforced by plain,
  deterministic Python functions instead.

A single Pydantic model, `SupportState`, is passed through the whole pipeline and can be
**paused and resumed** across multiple customer messages — this is what makes the
human-in-the-loop "ask the customer a clarifying question, then continue later" flow
possible.

**The single most important implementation detail in this notebook:** Colab/Jupyter
notebooks already run their own asyncio event loop, so every CrewAI execution here uses
`await crew.kickoff_async()` — never the synchronous `crew.kickoff()`, which would raise
a `RuntimeError` inside a notebook.

A markdown explanation cell has been inserted directly before each code cell below,
walking through it line by line. The original notebook's own explanatory markdown cells
(section headers, diagrams, troubleshooting notes, and the student-challenges list) have
all been preserved exactly as written, so you get both the author's own commentary and
this added line-by-line breakdown together.

**Key ideas to watch for as you go:**
1. **Structured output** (`output_pydantic=...`) lets CrewAI validate/parse LLM
   responses into typed Pydantic objects automatically — no manual `json.loads()`.
2. **The `@tool` gotcha**: a decorated function becomes a CrewAI `Tool` object, not a
   normal callable — this notebook keeps a plain "underscore" function
   (`_validate_refund`, `_process_refund`) alongside every `@tool`-wrapped version for
   application code to call directly.
3. **Deterministic routing and validation**: which agent runs next, and whether a
   refund is actually eligible, is decided by ordinary Python `if` statements and
   dictionary lookups — never by asking the LLM to decide for itself. This is the
   notebook's central "enterprise safety" lesson.
4. **Stateful pause/resume**: `SupportState` (plus the `existing_state` parameter on
   `support_request`) is how a multi-turn, clarification-needing conversation is
   modeled without a database — by simply returning and re-passing-in the same object.
5. **Async throughout**: every LLM-calling function is `async def ... await
   crew.kickoff_async()`, because notebooks already run inside an event loop.


# CrewAI Customer Support Swarm — Gemini
## Async-safe Colab/Jupyter implementation

End-to-end customer support system:

```text
Customer
   ↓
Conversation API
   ↓
Triage Agent
   ↓
Intent + Priority
   ↓
Routing Layer
   ├── Technical Support → Knowledge Base → Resolve / Escalate
   └── Refund Agent → Validate
                       ↓
                    Complete?
                    /      \
                  NO        YES
                  ↓          ↓
             Ask Customer  Process
                  ↓
          Customer Response
                  ↓
                Resume
```

**Important:** Colab/Jupyter already runs an asyncio event loop, so every CrewAI execution in this notebook uses `await crew.kickoff_async()`.

## Learning objectives

- CrewAI Agent, Task and Crew
- Gemini as the LLM
- Role, Goal and Backstory
- Tool calling
- Structured Pydantic outputs
- Sequential execution
- Deterministic routing
- Human-in-the-loop clarification
- Stateful pause/resume
- Technical escalation
- Refund validation and idempotency
- CrewAI vs LangGraph

### 🔎 Line-by-line — Install dependencies

```python
%pip install -q -U crewai litellm google-genai pydantic
```

- `%pip` is a **Jupyter magic command** — the `%` prefix tells Jupyter to run this as a
  notebook-aware pip install (safer than `!pip` because it installs into the *exact*
  kernel currently running, which matters a lot in Colab where multiple Python
  environments can exist).
- `install -q -U` — install/upgrade the listed packages; `-q` = quiet, `-U` = upgrade
  to latest.
- Four packages are installed:
  - **`crewai`** — the multi-agent framework this whole notebook is built on (`Agent`,
    `Task`, `Crew`, `Process`, `@tool`).
  - **`litellm`** — CrewAI uses LiteLLM under the hood as a universal adapter so you can
    point `llm=` at almost any provider (OpenAI, Gemini, Anthropic, etc.) using a simple
    string like `"gemini/gemini-2.5-flash"`.
  - **`google-genai`** — Google's SDK for calling Gemini models (LiteLLM uses this to
    actually talk to Google's API).
  - **`pydantic`** — used throughout the notebook to define strict, typed data models
    (`SupportState`, `TriageResult`, etc.) so LLM outputs can be validated instead of
    trusted blindly.

In [ ]:
%pip install -q -U crewai litellm google-genai pydantic

### 🔎 Line-by-line — Load the API key and configuration

```python
import os
from getpass import getpass

if not os.getenv("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Enter GEMINI_API_KEY: ")
```

- `import os` — lets us read/write environment variables.
- `from getpass import getpass` — imports a function that shows a **hidden** input box
  (like a password field), so the key isn't printed on screen or saved in the notebook
  file when you share it.
- `if not os.getenv("GEMINI_API_KEY"):` — `os.getenv(name)` returns the value of that
  environment variable, or `None` if it isn't set. `None` is "falsy" in Python, so
  `not os.getenv(...)` is `True` when the key **hasn't** already been set. This means:
  if you re-run this cell, and the key is already in memory, it **won't ask again**.
- `os.environ["GEMINI_API_KEY"] = getpass(...)` — only runs if the key was missing;
  prompts the user to paste their key, then stores it as an environment variable so any
  library that expects `GEMINI_API_KEY` (LiteLLM does, automatically) can find it.

```python
GEMINI_MODEL = "gemini/gemini-2.5-flash"
```

- A plain string constant. The `"gemini/..."` prefix is **LiteLLM's naming convention**:
  it tells LiteLLM which *provider* (`gemini`) and which *model*
  (`gemini-2.5-flash`) to route the request to. Every `Agent` below sets
  `llm=GEMINI_MODEL` to reuse this exact string.

```python
from typing import Optional, Literal
from pydantic import BaseModel, Field
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool

print("Configuration loaded.")
```

- `Optional`, `Literal` (from `typing`) — used to build precise type hints:
  - `Optional[str]` means "a string, or `None`."
  - `Literal["a", "b"]` restricts a field to *only* those exact values — this is how the
    notebook forces the LLM's structured output (e.g., `intent`) to be one of a fixed
    set of choices, rather than any arbitrary string.
- `BaseModel, Field` (from `pydantic`) — `BaseModel` is the base class every structured
  data model in this notebook inherits from (`SupportState`, `TriageResult`,
  `TechnicalResult`, `RefundIntake`). `Field` lets you customize a field's behavior,
  e.g. giving it a default value factory.
- `Agent, Task, Crew, Process` (from `crewai`) — the four core CrewAI building blocks:
  - **`Agent`** — a role-playing LLM persona (has a role, goal, backstory, tools).
  - **`Task`** — one unit of work an agent must complete, with a description and
    expected output.
  - **`Crew`** — a group of agents + tasks that execute together.
  - **`Process`** — controls *how* the crew executes its tasks (e.g., `Process.sequential`
    used throughout this notebook means tasks run one after another, in order).
- `tool` (from `crewai.tools`) — the decorator used to turn a plain Python function into
  a CrewAI `Tool` object an agent can call (explained in detail in the tools cell below).
- `print("Configuration loaded.")` — a simple confirmation message so you know this
  setup cell ran successfully before moving on.

In [ ]:
import os
from getpass import getpass

if not os.getenv("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Enter GEMINI_API_KEY: ")

GEMINI_MODEL = "gemini/gemini-2.5-flash"

from typing import Optional, Literal
from pydantic import BaseModel, Field
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool

print("Configuration loaded.")

## 1. Application state

The state survives while the workflow is waiting for customer clarification.

### 🔎 Line-by-line — Application state (`SupportState`)

```python
class SupportState(BaseModel):
    customer_id: str
    customer_message: str
```

- `class SupportState(BaseModel):` — defines a new **Pydantic model**. Inheriting from
  `BaseModel` gives this class automatic data validation: if you try to create a
  `SupportState` with the wrong type for a field, Pydantic raises a clear error
  immediately instead of letting bad data flow silently through the system.
- `customer_id: str` and `customer_message: str` — **required** fields (no default
  value given), meaning every `SupportState` must be created with both of these.

```python
    intent: Optional[Literal["technical", "refund", "general"]] = None
    priority: Optional[Literal["low", "medium", "high", "critical"]] = None
    triage_reason: Optional[str] = None
```

- These three fields start as `None` (not yet known) and get filled in **after** the
  Triage Agent runs.
- `Optional[Literal[...]]` combines both ideas: the value is either `None`, *or* one of
  the exact listed strings — nothing else is allowed. This is a strong safety net: even
  if the LLM tries to output some other word, Pydantic validation would reject it.

```python
    status: str = "NEW"
    response: Optional[str] = None
```

- `status: str = "NEW"` — a plain string field **with a default value**, so every new
  `SupportState` starts life in the `"NEW"` status. This field is the backbone of the
  whole workflow — it gets updated to values like `"TRIAGED"`, `"WAITING_FOR_CUSTOMER"`,
  `"RESOLVED"`, `"ESCALATED"`, `"REFUND_COMPLETED"`, etc. as the conversation progresses.
- `response` — the message that will eventually be shown back to the customer.

```python
    refund_order_id: Optional[str] = None
    refund_reason: Optional[str] = None
    refund_amount: Optional[float] = None
    missing_fields: list[str] = Field(default_factory=list)
```

- These four fields only matter for refund-intent conversations. They start empty and
  get filled in gradually as the customer provides more information across multiple
  messages (this is what makes the human-in-the-loop pause/resume flow possible).
- `missing_fields: list[str] = Field(default_factory=list)` — **important pattern**:
  you *cannot* safely write `= []` as a default value for a Pydantic (or dataclass)
  field, because that same list object could accidentally be shared/mutated across
  multiple instances. `Field(default_factory=list)` tells Pydantic to call `list()`
  fresh **every time** a new `SupportState` is created, guaranteeing each instance gets
  its own independent, empty list.

```python
    technical_solution: Optional[str] = None
    escalated: bool = False
    escalation_reason: Optional[str] = None
```

- These three fields only matter for technical-intent conversations: the solution text
  (if resolved), whether the case was escalated (`bool`, defaulting to `False`), and why.
- **Big picture:** this single `SupportState` object is what gets passed around, updated,
  and — critically — **returned to the caller and passed back in later** so the
  conversation can pause (waiting on the customer) and resume exactly where it left off,
  without losing any previously-gathered information.

In [ ]:
class SupportState(BaseModel):
    customer_id: str
    customer_message: str

    intent: Optional[Literal["technical", "refund", "general"]] = None
    priority: Optional[Literal["low", "medium", "high", "critical"]] = None
    triage_reason: Optional[str] = None

    status: str = "NEW"
    response: Optional[str] = None

    refund_order_id: Optional[str] = None
    refund_reason: Optional[str] = None
    refund_amount: Optional[float] = None
    missing_fields: list[str] = Field(default_factory=list)

    technical_solution: Optional[str] = None
    escalated: bool = False
    escalation_reason: Optional[str] = None

## 2. Mock enterprise systems

These simulate order/payment data and a technical knowledge base.

### 🔎 Line-by-line — Mock enterprise systems

```python
ORDERS = {
    "ORD-1001": {
        "customer_id": "CUST-1001",
        "amount": 1499.0,
        "payment_status": "failed",
        "refund_status": "not_requested",
        "product": "Wireless Headphones",
    },
    "ORD-1002": {
        "customer_id": "CUST-1002",
        "amount": 2999.0,
        "payment_status": "captured",
        "refund_status": "not_requested",
        "product": "Mechanical Keyboard",
    },
}
```

- `ORDERS` is a plain Python dictionary acting as a **fake database/order system** — in
  a real production system this would be a call to an Orders microservice or SQL table
  instead. Each key is an order ID (`"ORD-1001"`); each value is a nested dict of order
  details: which customer owns it, its amount, payment status, refund status, and
  product name.
- Notice `ORD-1001` belongs to `CUST-1001` with `payment_status: "failed"`, while
  `ORD-1002` belongs to `CUST-1002` with `payment_status: "captured"` — these specific
  values are deliberately chosen to drive the different scenarios demonstrated later in
  the notebook (eligible vs. not-eligible refunds, mismatched customer/order pairs).

```python
KB = {
    "payment failed": (
        "If payment failed but money was deducted, verify the payment transaction. "
        "A temporary authorization may reverse automatically."
    ),
    "login": (
        "Verify credentials, reset the password, and check MFA. "
        "If the problem persists, escalate with timestamp and account ID."
    ),
    "app crash": (
        "Collect application version, device/OS, timestamp and error message. "
        "Restart and update the application. Escalate recurring crashes with logs."
    ),
}
```

- `KB` (Knowledge Base) is another plain dictionary — a **fake technical support
  article database**. Each key is a short topic phrase; each value is a
  (parenthesized, auto-concatenated) string of guidance text for that topic. In Python,
  two string literals written next to each other inside `( ... )` are automatically
  joined into one string at compile time — this is just a readable way to write a long
  string across multiple lines without needing `\n` or `+`.
- This dictionary is what the `search_knowledge_base` tool (defined in the next cell)
  searches through when the Technical Agent needs to look something up.

```python
REFUND_POLICY = {
    "eligible_payment_statuses": {"failed", "captured"},
}

print("Mock enterprise systems loaded.")
```

- `REFUND_POLICY` — a small config dictionary encoding a **business rule**: which
  payment statuses are allowed to be refunded. Notice the value is written with
  `{ }` containing comma-separated items with no `:` — that makes it a Python **set**
  (not a dict), used later for fast membership testing (`status in REFUND_POLICY[...]`).
- This is a deliberate design choice worth highlighting: refund *eligibility rules* live
  in plain Python data structures, not inside an LLM prompt — a core theme of this
  notebook is keeping financial/business logic **deterministic** and outside the LLM's
  control (more on this in the tools cell).

In [ ]:
ORDERS = {
    "ORD-1001": {
        "customer_id": "CUST-1001",
        "amount": 1499.0,
        "payment_status": "failed",
        "refund_status": "not_requested",
        "product": "Wireless Headphones",
    },
    "ORD-1002": {
        "customer_id": "CUST-1002",
        "amount": 2999.0,
        "payment_status": "captured",
        "refund_status": "not_requested",
        "product": "Mechanical Keyboard",
    },
}

KB = {
    "payment failed": (
        "If payment failed but money was deducted, verify the payment transaction. "
        "A temporary authorization may reverse automatically."
    ),
    "login": (
        "Verify credentials, reset the password, and check MFA. "
        "If the problem persists, escalate with timestamp and account ID."
    ),
    "app crash": (
        "Collect application version, device/OS, timestamp and error message. "
        "Restart and update the application. Escalate recurring crashes with logs."
    ),
}

REFUND_POLICY = {
    "eligible_payment_statuses": {"failed", "captured"},
}

print("Mock enterprise systems loaded.")

## 3. CrewAI tools

### Important `@tool` detail

CrewAI's `@tool` decorator turns the decorated function into a CrewAI `Tool` object.

Therefore the application layer should not call the decorated object directly:

```python
validate_refund(...)       # DON'T do this
```

Instead this notebook uses:

```text
Agent → CrewAI Tool → _validate_refund()
Application → _validate_refund()
```

This prevents:

```text
TypeError: 'Tool' object is not callable
```

while still exposing the tools to the Refund Agent.


### 🔎 Line-by-line — CrewAI tools (and the `@tool` gotcha)

The markdown cell above this one explains an important CrewAI trap: the `@tool`
decorator turns a plain function into a CrewAI `Tool` **object**, which is no longer
directly callable like a normal Python function. This cell's structure is built
specifically to work around that.

```python
@tool("search_knowledge_base")
def search_knowledge_base(query: str) -> str:
    """Search the mock technical support knowledge base."""
    q = query.lower()
    matches = []
    for key, value in KB.items():
        if key in q or any(word in q for word in key.split()):
            matches.append(f"{key}: {value}")
    return "\n".join(matches) if matches else "No matching knowledge-base article was found."
```

- `@tool("search_knowledge_base")` — a decorator: it wraps the function immediately
  below it and turns it into a CrewAI `Tool` named `"search_knowledge_base"`. This name
  (and the function's docstring) is what the LLM agent actually **sees** when deciding
  whether/how to call this tool — so the docstring isn't just documentation, it's part
  of the prompt the agent uses to decide when this tool is useful.
- `q = query.lower()` — normalizes the search query to lowercase so matching isn't
  case-sensitive.
- `for key, value in KB.items():` — loops through every topic in the knowledge base.
- `if key in q or any(word in q for word in key.split()):` — a simple substring-match
  search: does the *whole* KB key phrase appear in the query, **or** does *any
  individual word* from that key phrase appear in the query? (`key.split()` breaks
  `"payment failed"` into `["payment", "failed"]`, so a query mentioning just
  `"payment"` would still match.) This is a basic keyword search — in a production
  system you'd likely replace this with a real search index or vector database (this is
  literally suggested as a "student challenge" later in the notebook).
- `matches.append(f"{key}: {value}")` — collect any matching articles formatted as
  `"topic: guidance text"`.
- The final line returns all matches joined by newlines, or a friendly "not found"
  message if `matches` is empty.

```python
@tool("lookup_order")
def lookup_order(order_id: str) -> str:
    """Look up a mock order by ID."""
    order = ORDERS.get(order_id.strip())
    return str(order) if order else f"Order {order_id} was not found."
```

- Another CrewAI tool, this time letting the Refund Agent look up order details.
- `order_id.strip()` removes any stray leading/trailing whitespace the LLM might have
  included when it generated the tool call argument.
- `ORDERS.get(...)` returns `None` if the ID isn't found (safer than `ORDERS[...]`,
  which would raise a `KeyError`).
- `str(order)` converts the order's dictionary into a text representation the LLM can
  read; if no order was found, a clear "not found" message is returned instead.

```python
def _validate_refund(order_id: str, customer_id: str) -> str:
    """Plain application-layer refund validation."""
    order = ORDERS.get(order_id.strip())
    if not order:
        return f"NOT_FOUND: Order {order_id} does not exist."
    if order["customer_id"] != customer_id:
        return "NOT_ELIGIBLE: Customer does not match the order."
    if order["refund_status"] != "not_requested":
        return f"NOT_ELIGIBLE: Refund status is {order['refund_status']}."
    if order["payment_status"] not in REFUND_POLICY["eligible_payment_statuses"]:
        return f"NOT_ELIGIBLE: Payment status {order['payment_status']} is not eligible."
    return f"ELIGIBLE: Amount={order['amount']}, Product={order['product']}"
```

- **This is the key deterministic-safety function of the whole notebook** — notice it's
  a *plain function*, **not** decorated with `@tool`. It's meant to be called directly
  by application code (not by the LLM), enforcing refund eligibility rules with 100%
  predictable logic that no prompt-injection or LLM mistake can bypass.
- It checks four conditions **in order**, returning early (`return`) as soon as one
  fails — a common "guard clause" pattern that keeps the function readable:
  1. Does the order exist at all? (`NOT_FOUND` if missing.)
  2. Does the `customer_id` on the order match the customer making the request?
     (`NOT_ELIGIBLE` if it's someone else's order — this is the authorization check
     demonstrated in Scenario E later.)
  3. Has a refund already been requested/processed for this order? (`NOT_ELIGIBLE` if
     it's not in the fresh `"not_requested"` state — this enforces **idempotency**,
     preventing double refunds.)
  4. Is the payment status one of the policy-approved statuses (`REFUND_POLICY`)?
     (`NOT_ELIGIBLE` otherwise.)
  5. If **all** checks pass, return an `"ELIGIBLE: ..."` string with the refund amount
     and product name.
- Every return value is a plain string with a clear, parseable prefix
  (`NOT_FOUND:`, `NOT_ELIGIBLE:`, `ELIGIBLE:`) — this makes it trivial for downstream
  code to check `.startswith("ELIGIBLE:")` rather than needing to parse complex
  structured data.

```python
def _process_refund(order_id: str, customer_id: str) -> str:
    """Plain application-layer refund operation."""
    order = ORDERS.get(order_id.strip())
    if not order:
        return "FAILED: Order not found."
    if order["customer_id"] != customer_id:
        return "FAILED: Customer/order mismatch."
    if order["refund_status"] != "not_requested":
        return f"FAILED: Refund already has status {order['refund_status']}."
    order["refund_status"] = "processed"
    return f"SUCCESS: Refund processed for {order['amount']} for order {order_id}."
```

- The actual "do the refund" function — again a **plain function**, not an `@tool`,
  and again re-checks the same safety conditions independently (never assume validation
  already happened just because another function was called earlier — always
  re-verify right before performing the irreversible action).
- `order["refund_status"] = "processed"` — this line **mutates the shared `ORDERS`
  dictionary in place**, which is what makes the idempotency check in `_validate_refund`
  work correctly on any future attempt to refund the same order again.
- Returns a `"SUCCESS: ..."` or `"FAILED: ..."` string, following the same
  parseable-prefix pattern as `_validate_refund`.

```python
@tool("validate_refund")
def validate_refund(order_id: str, customer_id: str) -> str:
    """Validate refund eligibility using deterministic business rules."""
    return _validate_refund(order_id, customer_id)


@tool("process_refund")
def process_refund(order_id: str, customer_id: str) -> str:
    """Process a mock refund after deterministic validation."""
    return _process_refund(order_id, customer_id)
```

- These two are **thin `@tool` wrappers** around the plain functions above them. This
  is the resolution to the `@tool` gotcha explained at the top of this section:
  - `validate_refund` / `process_refund` (no underscore) are CrewAI `Tool` objects —
    these are what get passed into `Agent(tools=[...])` so the **LLM agent** can call
    them during its reasoning.
  - `_validate_refund` / `_process_refund` (underscore prefix) are ordinary Python
    functions — these are what the **application code** (e.g. `handle_refund`, seen
    later) calls directly, since calling the `@tool`-wrapped versions directly would
    raise `TypeError: 'Tool' object is not callable`.
  - The underscore prefix is a common Python convention meaning "internal/private —
    not meant to be used from outside this module," which fits perfectly here since
    these are implementation details the tool wrappers delegate to.

In [ ]:
@tool("search_knowledge_base")
def search_knowledge_base(query: str) -> str:
    """Search the mock technical support knowledge base."""
    q = query.lower()
    matches = []
    for key, value in KB.items():
        if key in q or any(word in q for word in key.split()):
            matches.append(f"{key}: {value}")
    return "\n".join(matches) if matches else "No matching knowledge-base article was found."


@tool("lookup_order")
def lookup_order(order_id: str) -> str:
    """Look up a mock order by ID."""
    order = ORDERS.get(order_id.strip())
    return str(order) if order else f"Order {order_id} was not found."


def _validate_refund(order_id: str, customer_id: str) -> str:
    """Plain application-layer refund validation."""
    order = ORDERS.get(order_id.strip())
    if not order:
        return f"NOT_FOUND: Order {order_id} does not exist."
    if order["customer_id"] != customer_id:
        return "NOT_ELIGIBLE: Customer does not match the order."
    if order["refund_status"] != "not_requested":
        return f"NOT_ELIGIBLE: Refund status is {order['refund_status']}."
    if order["payment_status"] not in REFUND_POLICY["eligible_payment_statuses"]:
        return f"NOT_ELIGIBLE: Payment status {order['payment_status']} is not eligible."
    return f"ELIGIBLE: Amount={order['amount']}, Product={order['product']}"


def _process_refund(order_id: str, customer_id: str) -> str:
    """Plain application-layer refund operation."""
    order = ORDERS.get(order_id.strip())
    if not order:
        return "FAILED: Order not found."
    if order["customer_id"] != customer_id:
        return "FAILED: Customer/order mismatch."
    if order["refund_status"] != "not_requested":
        return f"FAILED: Refund already has status {order['refund_status']}."
    order["refund_status"] = "processed"
    return f"SUCCESS: Refund processed for {order['amount']} for order {order_id}."


@tool("validate_refund")
def validate_refund(order_id: str, customer_id: str) -> str:
    """Validate refund eligibility using deterministic business rules."""
    return _validate_refund(order_id, customer_id)


@tool("process_refund")
def process_refund(order_id: str, customer_id: str) -> str:
    """Process a mock refund after deterministic validation."""
    return _process_refund(order_id, customer_id)

## 4. Create the three role-based agents

### 🔎 Line-by-line — Creating the three role-based agents

```python
triage_agent = Agent(
    role="Customer Support Triage Specialist",
    goal="Classify customer requests as technical, refund, or general and assign an appropriate priority.",
    backstory="You are first-line support intelligence for a large enterprise. Be precise and never invent customer facts.",
    llm=GEMINI_MODEL,
    verbose=True,
    allow_delegation=False,
)
```

- `Agent(...)` constructs a CrewAI agent — think of it as a configured LLM "persona"
  with a specific job. Each parameter shapes how the agent behaves:
  - **`role`** — a short label describing *who* this agent is; this text is injected
    into the underlying prompt CrewAI builds, helping the LLM stay "in character."
  - **`goal`** — what this agent is trying to accomplish; also injected into the
    prompt, keeping the agent focused on its specific job (classification here) rather
    than trying to do everything.
  - **`backstory`** — additional context/personality that further shapes the agent's
    behavior. Notice it explicitly says *"never invent customer facts"* — this is a
    prompt-engineering guardrail against hallucination, baked directly into the agent's
    identity rather than repeated in every task.
  - **`llm=GEMINI_MODEL`** — tells this agent to use the Gemini model string defined
    earlier (`"gemini/gemini-2.5-flash"`), routed through LiteLLM.
  - **`verbose=True`** — makes CrewAI print out its internal reasoning/tool-call steps
    as it runs, which is very useful for learning/debugging (you can watch the agent
    "think").
  - **`allow_delegation=False`** — by default, CrewAI agents can delegate sub-tasks to
    other agents in the crew; this is explicitly turned **off** here, since each of
    these three agents in this notebook is meant to work independently within its own
    dedicated `Crew` (delegation isn't used anywhere in this design).
- Notice this agent has **no `tools=[...]` parameter** — the Triage Agent doesn't need
  to look anything up externally; it only needs to *read and classify* the message
  itself.

```python
technical_agent = Agent(
    role="Senior Technical Support Engineer",
    goal="Diagnose technical issues using the knowledge base and safely resolve or escalate them.",
    backstory="You are an experienced production support engineer. Never invent a fix when evidence is insufficient.",
    tools=[search_knowledge_base],
    llm=GEMINI_MODEL,
    verbose=True,
    allow_delegation=False,
)
```

- Same overall pattern, but this agent is given `tools=[search_knowledge_base]` — a
  **list** containing the one CrewAI tool it's allowed to call. This is how CrewAI
  exposes function-calling to an agent: during its reasoning, the LLM can decide to
  invoke `search_knowledge_base(query=...)`, receive the returned text back into its
  context, and continue reasoning with that new information.
- The backstory again bakes in an anti-hallucination instruction specific to this
  agent's job: *"Never invent a fix when evidence is insufficient."*

```python
refund_agent = Agent(
    role="Senior Refund Operations Specialist",
    goal="Validate refund requests, identify missing information, and process only validated refunds.",
    backstory="You are a careful refund specialist. Financial actions require verification. Never fabricate order IDs, amounts, or eligibility.",
    tools=[lookup_order, validate_refund, process_refund],
    llm=GEMINI_MODEL,
    verbose=True,
    allow_delegation=False,
)

print("Agents created.")
```

- This agent gets **three** tools — it can look up an order, validate refund
  eligibility, and (if eligible) process the refund, all via LLM-initiated tool calls.
- Note: even though this agent *can* call `validate_refund`/`process_refund` as tools,
  you'll see later (in `handle_refund`) that the **application code deliberately calls
  the underscore-prefixed plain functions directly instead**, rather than relying on the
  agent to decide to call these tools on its own. This is a deliberate architectural
  choice: financial actions are too important to leave entirely to LLM judgment about
  *whether* to call a tool — the notebook uses the LLM only for information
  *extraction*, while the actual validate/process decision is forced deterministically
  by Python code. This is discussed more directly in the `handle_refund` cell below.
- `print("Agents created.")` confirms the cell ran successfully.

In [ ]:
triage_agent = Agent(
    role="Customer Support Triage Specialist",
    goal="Classify customer requests as technical, refund, or general and assign an appropriate priority.",
    backstory="You are first-line support intelligence for a large enterprise. Be precise and never invent customer facts.",
    llm=GEMINI_MODEL,
    verbose=True,
    allow_delegation=False,
)

technical_agent = Agent(
    role="Senior Technical Support Engineer",
    goal="Diagnose technical issues using the knowledge base and safely resolve or escalate them.",
    backstory="You are an experienced production support engineer. Never invent a fix when evidence is insufficient.",
    tools=[search_knowledge_base],
    llm=GEMINI_MODEL,
    verbose=True,
    allow_delegation=False,
)

refund_agent = Agent(
    role="Senior Refund Operations Specialist",
    goal="Validate refund requests, identify missing information, and process only validated refunds.",
    backstory="You are a careful refund specialist. Financial actions require verification. Never fabricate order IDs, amounts, or eligibility.",
    tools=[lookup_order, validate_refund, process_refund],
    llm=GEMINI_MODEL,
    verbose=True,
    allow_delegation=False,
)

print("Agents created.")

## 5. Triage — ASYNC

The key fix for the Colab/Jupyter event-loop error is `await crew.kickoff_async()`.

### 🔎 Line-by-line — Triage (async)

```python
class TriageResult(BaseModel):
    intent: Literal["technical", "refund", "general"]
    priority: Literal["low", "medium", "high", "critical"]
    reason: str
```

- A small Pydantic model describing **exactly** the shape of data the Triage Agent must
  return: one of three fixed intents, one of four fixed priority levels, and a
  free-text `reason`. Because these use `Literal` (not `Optional[Literal]` this time),
  all three fields are **required** — the LLM's output must include all of them or
  validation will fail.

```python
async def run_triage(message: str) -> TriageResult:
    task = Task(
        description=(
            "Classify the customer request. "
            "technical = technical/application/device/API problem; "
            "refund = money-back/refund/cancellation-with-refund/duplicate-charge; "
            "general = everything else. "
            "Assign low, medium, high, or critical priority. "
            "Never invent facts.\n\n"
            f"Customer message:\n{message}"
        ),
        expected_output="A valid TriageResult JSON object.",
        agent=triage_agent,
        output_pydantic=TriageResult,
    )
```

- `async def run_triage(message: str) -> TriageResult:` — this function is declared
  **`async`**, meaning it must be called with `await` (never called normally). This
  matters specifically because Colab/Jupyter notebooks already run inside their own
  asyncio event loop, and CrewAI's async execution method needs to cooperate with that
  loop rather than trying to start a new one (the markdown cell above explains this is
  "the key fix" for a common Colab error).
- `Task(...)` defines one unit of work for an agent to perform:
  - **`description`** — the actual instructions for this task, built as a Python string
    (the outer parentheses `( ... )` let multiple string literals sit on separate
    lines and get auto-concatenated, same trick seen in the `KB` dictionary earlier).
    It defines each intent category with clear criteria, asks for a priority, repeats
    the "never invent facts" guardrail, then appends the actual customer `message` via
    an f-string.
  - **`expected_output`** — a plain-English description of what the output should look
    like; this text also gets included in the prompt CrewAI builds, further nudging the
    LLM toward the right output shape.
  - **`agent=triage_agent`** — which agent is responsible for completing this task.
  - **`output_pydantic=TriageResult`** — this is the critical line that makes
    **structured output** possible: it tells CrewAI to validate/parse the LLM's raw text
    response into a `TriageResult` object automatically, rather than leaving you to
    parse JSON by hand (contrast this with the earlier LangGraph notebook, which had to
    manually call `json.loads()`).

```python
    crew = Crew(
        agents=[triage_agent],
        tasks=[task],
        process=Process.sequential,
        verbose=True,
    )

    result = await crew.kickoff_async()
    return result.pydantic
```

- `Crew(agents=[...], tasks=[...], process=Process.sequential, verbose=True)` — a
  `Crew` bundles agents and tasks together. Here there's only one agent and one task,
  but the same pattern scales to multi-agent crews.
  - `process=Process.sequential` — tasks execute one after another, in the order given
    (as opposed to `Process.hierarchical`, where a manager agent could dynamically
    delegate — not used in this notebook).
- `result = await crew.kickoff_async()` — **this is the critical async pattern
  repeated throughout the notebook.** `crew.kickoff_async()` starts the crew's
  execution (running the LLM, handling any tool calls, etc.) as an asyncio coroutine;
  `await` pauses this function until that work completes, without blocking the
  notebook's already-running event loop. Using the *synchronous* `crew.kickoff()`
  instead would raise a `RuntimeError` inside Colab/Jupyter, because you can't start a
  second nested event loop from within one that's already running — this exact error is
  documented in the troubleshooting section near the end of the notebook.
- `return result.pydantic` — `crew.kickoff_async()` returns a `CrewOutput` object;
  because the task specified `output_pydantic=TriageResult`, the `.pydantic` attribute
  gives you back an already-validated `TriageResult` instance directly, ready to use in
  regular Python code (`.intent`, `.priority`, `.reason`).

In [ ]:
class TriageResult(BaseModel):
    intent: Literal["technical", "refund", "general"]
    priority: Literal["low", "medium", "high", "critical"]
    reason: str


async def run_triage(message: str) -> TriageResult:
    task = Task(
        description=(
            "Classify the customer request. "
            "technical = technical/application/device/API problem; "
            "refund = money-back/refund/cancellation-with-refund/duplicate-charge; "
            "general = everything else. "
            "Assign low, medium, high, or critical priority. "
            "Never invent facts.\n\n"
            f"Customer message:\n{message}"
        ),
        expected_output="A valid TriageResult JSON object.",
        agent=triage_agent,
        output_pydantic=TriageResult,
    )

    crew = Crew(
        agents=[triage_agent],
        tasks=[task],
        process=Process.sequential,
        verbose=True,
    )

    result = await crew.kickoff_async()
    return result.pydantic

## 6. Technical Support — ASYNC

### 🔎 Line-by-line — Technical support (async)

```python
class TechnicalResult(BaseModel):
    resolved: bool
    answer: str
    escalation_reason: Optional[str] = None
```

- Another structured-output schema, this time for the Technical Agent's decision:
  - `resolved: bool` — did the agent find a safe, confident fix?
  - `answer: str` — the customer-facing explanation (used either way — as the solution
    text if resolved, or context if not).
  - `escalation_reason: Optional[str] = None` — only meaningful when `resolved=False`;
    explains *why* human escalation is needed.

```python
async def run_technical_support(message: str) -> TechnicalResult:
    task = Task(
        description=(
            "Resolve this technical support request:\n\n"
            f"{message}\n\n"
            "Use the knowledge base before answering. "
            "If a safe solution is available, provide concise customer-facing steps. "
            "If evidence is insufficient, set resolved=false and explain why escalation is needed."
        ),
        expected_output="A valid TechnicalResult JSON object.",
        agent=technical_agent,
        output_pydantic=TechnicalResult,
    )
```

- Same `Task(...)` pattern as the triage task, but instructing the Technical Agent to
  **use its tool** (`search_knowledge_base`, attached back when this agent was created)
  *before* attempting to answer — this nudges the agent toward grounding its answer in
  the actual KB content rather than guessing from its own training knowledge.
- The instruction explicitly gives the agent an **honest failure path**: if the
  knowledge base doesn't have enough information, it should say so (`resolved=false`)
  rather than inventing a plausible-sounding but potentially wrong fix — this is what
  drives Scenario D (technical escalation) later in the notebook.

```python
    crew = Crew(
        agents=[technical_agent],
        tasks=[task],
        process=Process.sequential,
        verbose=True,
    )

    result = await crew.kickoff_async()
    return result.pydantic
```

- Identical crew-setup and async-execution pattern as `run_triage` — by this point in
  the notebook you should recognize this as the **reusable template** every "run an
  agent on one task" function follows: build a `Task`, wrap it in a single-agent
  `Crew`, `await crew.kickoff_async()`, return `result.pydantic`.

In [ ]:
class TechnicalResult(BaseModel):
    resolved: bool
    answer: str
    escalation_reason: Optional[str] = None


async def run_technical_support(message: str) -> TechnicalResult:
    task = Task(
        description=(
            "Resolve this technical support request:\n\n"
            f"{message}\n\n"
            "Use the knowledge base before answering. "
            "If a safe solution is available, provide concise customer-facing steps. "
            "If evidence is insufficient, set resolved=false and explain why escalation is needed."
        ),
        expected_output="A valid TechnicalResult JSON object.",
        agent=technical_agent,
        output_pydantic=TechnicalResult,
    )

    crew = Crew(
        agents=[technical_agent],
        tasks=[task],
        process=Process.sequential,
        verbose=True,
    )

    result = await crew.kickoff_async()
    return result.pydantic

## 7. Refund intake — ASYNC + HUMAN-IN-THE-LOOP

The Refund Agent first extracts information.

If the order ID or reason is missing:

```text
Refund Agent
     ↓
Missing information
     ↓
Ask customer
     ↓
WAITING_FOR_CUSTOMER
```

No refund is processed at this point.

### 🔎 Line-by-line — Refund intake (async + human-in-the-loop)

```python
class RefundIntake(BaseModel):
    order_id: Optional[str] = None
    reason: Optional[str] = None
    amount: Optional[float] = None
    missing_fields: list[str] = Field(default_factory=list)
    clarification_question: Optional[str] = None
```

- This schema represents the *information-extraction* step of a refund request — every
  field is optional because, on a customer's first message, some of this information
  might simply not be present yet.
- `missing_fields` uses the same `Field(default_factory=list)` safe-default pattern
  seen in `SupportState`.
- `clarification_question` — if information is missing, this holds the single question
  the agent wants to ask the customer next (used to drive the human-in-the-loop pause).

```python
async def extract_refund_details(
    message: str,
    previous_state: SupportState,
) -> RefundIntake:

    task = Task(
        description=(
            "Extract refund information from the current customer message.\n\n"
            f"Current message: {message}\n"
            f"Existing order ID: {previous_state.refund_order_id}\n"
            f"Existing reason: {previous_state.refund_reason}\n"
            f"Existing amount: {previous_state.refund_amount}\n\n"
            "Required information: order_id and reason. "
            "Amount can be obtained from order lookup when an order ID exists. "
            "Never invent missing information. "
            "If required information is missing, list the missing fields "
            "and ask one concise clarification question."
        ),
        expected_output="A valid RefundIntake JSON object.",
        agent=refund_agent,
        output_pydantic=RefundIntake,
    )
```

- `extract_refund_details` takes **two** arguments: the new `message` from the customer,
  and `previous_state` — the entire `SupportState` accumulated so far. This is the key
  mechanism that makes multi-turn clarification work: the prompt includes *both* what
  the customer just said *and* whatever order ID/reason/amount were already captured
  from earlier in the conversation (`previous_state.refund_order_id`, etc.), so the LLM
  can combine new and old information rather than starting from scratch every turn.
- The task explicitly tells the agent: `"Never invent missing information."` and to
  produce, if something is still missing, both a list of `missing_fields` **and** a
  single, concise `clarification_question` — directly matching the `RefundIntake`
  schema's fields.
- Same `Task(..., output_pydantic=RefundIntake)` structured-output pattern as before.

```python
    crew = Crew(
        agents=[refund_agent],
        tasks=[task],
        process=Process.sequential,
        verbose=True,
    )

    result = await crew.kickoff_async()
    return result.pydantic
```

- The same reusable single-agent-crew-and-await template seen in every "run one agent"
  function so far.

In [ ]:
class RefundIntake(BaseModel):
    order_id: Optional[str] = None
    reason: Optional[str] = None
    amount: Optional[float] = None
    missing_fields: list[str] = Field(default_factory=list)
    clarification_question: Optional[str] = None


async def extract_refund_details(
    message: str,
    previous_state: SupportState,
) -> RefundIntake:

    task = Task(
        description=(
            "Extract refund information from the current customer message.\n\n"
            f"Current message: {message}\n"
            f"Existing order ID: {previous_state.refund_order_id}\n"
            f"Existing reason: {previous_state.refund_reason}\n"
            f"Existing amount: {previous_state.refund_amount}\n\n"
            "Required information: order_id and reason. "
            "Amount can be obtained from order lookup when an order ID exists. "
            "Never invent missing information. "
            "If required information is missing, list the missing fields "
            "and ask one concise clarification question."
        ),
        expected_output="A valid RefundIntake JSON object.",
        agent=refund_agent,
        output_pydantic=RefundIntake,
    )

    crew = Crew(
        agents=[refund_agent],
        tasks=[task],
        process=Process.sequential,
        verbose=True,
    )

    result = await crew.kickoff_async()
    return result.pydantic

## 8. Refund handler — ASYNC

### 🔎 Line-by-line — Refund handler (async)

```python
async def handle_refund(state: SupportState) -> SupportState:
    intake = await extract_refund_details(
        state.customer_message,
        state,
    )
```

- `handle_refund` takes the entire current `state` and returns an updated version of it
  — this "take state in, return updated state out" pattern is used consistently
  throughout the notebook, which makes it easy to reason about each function as a pure
  transformation step.
- First, it calls the LLM-based extraction function from the previous cell, passing in
  the latest customer message *and* the full previous state (so prior refund details
  aren't lost).

```python
    if intake.order_id:
        state.refund_order_id = intake.order_id

    if intake.reason:
        state.refund_reason = intake.reason

    if intake.amount is not None:
        state.refund_amount = intake.amount

    state.missing_fields = intake.missing_fields
```

- These four `if` blocks **merge** the newly extracted `intake` fields into the
  persistent `state`, but only overwrite a field if the new extraction actually found
  something (`if intake.order_id:` is falsy for both `None` and empty string, so a
  blank extraction won't erase previously-known information).
- `if intake.amount is not None:` uses an explicit `is not None` check rather than a
  plain truthiness check — this is deliberate, because `0.0` (a legitimate possible
  refund amount, even if unlikely) would be falsy under a plain `if intake.amount:`
  check and would be incorrectly skipped. This is a subtle but important bug-avoidance
  pattern worth pointing out to students.
- `state.missing_fields = intake.missing_fields` — always overwritten (not
  conditionally merged), since this list should always reflect the *current* extraction
  attempt's view of what's still missing.

```python
    if intake.missing_fields:
        state.status = "WAITING_FOR_CUSTOMER"
        state.response = intake.clarification_question
        return state
```

- **This is the human-in-the-loop pause.** If anything is still missing after merging
  in the new extraction, the function immediately sets `status` to
  `"WAITING_FOR_CUSTOMER"`, sets the outgoing `response` to the agent's clarification
  question, and **returns early** — deliberately *not* attempting any validation or
  refund processing yet. The function stops here and control returns all the way back
  up to whoever is driving the conversation (a human, an API caller, etc.), who is
  expected to collect the customer's answer and call back in later with this same
  `state` object plus the new message.

```python
    validation = _validate_refund(
        state.refund_order_id,
        state.customer_id,
    )
```

- If execution reaches this point, all required information is present. Notice this
  calls **`_validate_refund`** (the underscore/plain-function version), **not** the
  `@tool`-wrapped `validate_refund` — because, as covered earlier, calling the `@tool`
  version directly from Python would raise a `TypeError`. This is also a deliberate
  design decision: the *decision* to validate happens in deterministic application
  code, not by hoping the LLM chooses to call the right tool at the right time.

```python
    if validation.startswith("ELIGIBLE:"):
        result = _process_refund(
            state.refund_order_id,
            state.customer_id,
        )

        state.status = (
            "REFUND_COMPLETED"
            if result.startswith("SUCCESS")
            else "REFUND_FAILED"
        )
        state.response = result
        return state

    state.status = "REFUND_REJECTED"
    state.response = validation
    return state
```

- `if validation.startswith("ELIGIBLE:"):` — checks the string prefix returned by
  `_validate_refund` (recall its return values always start with `ELIGIBLE:`,
  `NOT_ELIGIBLE:`, or `NOT_FOUND:`).
- If eligible, it calls `_process_refund` (again, the plain function, not the tool) to
  actually perform the refund.
- `state.status = ("REFUND_COMPLETED" if result.startswith("SUCCESS") else "REFUND_FAILED")`
  — a conditional expression (Python's inline if/else) that sets the final status based
  on whether processing actually succeeded — note that even after passing validation,
  processing could still fail for other reasons, so this is checked independently
  rather than assumed.
- If the original validation was *not* eligible, `status` is set to `"REFUND_REJECTED"`
  and the `response` is simply the raw validation message (e.g.
  `"NOT_ELIGIBLE: Customer does not match the order."`) — this is exactly what drives
  Scenario E (customer/order mismatch) later in the notebook.

In [ ]:
async def handle_refund(state: SupportState) -> SupportState:
    intake = await extract_refund_details(
        state.customer_message,
        state,
    )

    if intake.order_id:
        state.refund_order_id = intake.order_id

    if intake.reason:
        state.refund_reason = intake.reason

    if intake.amount is not None:
        state.refund_amount = intake.amount

    state.missing_fields = intake.missing_fields

    if intake.missing_fields:
        state.status = "WAITING_FOR_CUSTOMER"
        state.response = intake.clarification_question
        return state

    validation = _validate_refund(
        state.refund_order_id,
        state.customer_id,
    )

    if validation.startswith("ELIGIBLE:"):
        result = _process_refund(
            state.refund_order_id,
            state.customer_id,
        )

        state.status = (
            "REFUND_COMPLETED"
            if result.startswith("SUCCESS")
            else "REFUND_FAILED"
        )
        state.response = result
        return state

    state.status = "REFUND_REJECTED"
    state.response = validation
    return state

## 9. Deterministic routing layer — ASYNC

### 🔎 Line-by-line — Deterministic routing layer (async)

```python
async def route_customer(state: SupportState) -> SupportState:

    if state.intent == "technical":
        technical = await run_technical_support(
            state.customer_message
        )

        if technical.resolved:
            state.status = "RESOLVED"
            state.response = technical.answer
            state.technical_solution = technical.answer
        else:
            state.status = "ESCALATED"
            state.escalated = True
            state.escalation_reason = technical.escalation_reason
            state.response = (
                "I could not safely resolve this issue automatically. "
                "Your request has been escalated to a support specialist."
            )
```

- `route_customer` is the **deterministic dispatcher**: notice it's a plain Python
  `if/elif/else` chain based on `state.intent` — **not** another LLM call. This is the
  architectural centerpiece of the "deterministic routing" learning objective: once the
  Triage Agent has classified the intent, *where the conversation goes next* is decided
  by ordinary, 100% predictable code, not by asking an LLM to route itself again.
- If `intent == "technical"`, it calls `run_technical_support` (async, so `await`ed)
  with just the customer's message.
- `if technical.resolved:` — reads the structured `TechnicalResult` returned from that
  call. If resolved, mark the conversation `"RESOLVED"` and copy the agent's answer into
  both the outward-facing `response` and the internal `technical_solution` record.
- Otherwise, mark it `"ESCALATED"`, flip `escalated = True`, store *why* it was
  escalated (from the agent's own `escalation_reason`), and give the customer a
  reassuring, honest response rather than a fabricated fix.

```python
    elif state.intent == "refund":
        state = await handle_refund(state)

    else:
        state.status = "GENERAL_SUPPORT"
        state.response = (
            "A general support representative will review your request."
        )

    return state
```

- If `intent == "refund"`, delegate the entire refund flow to `handle_refund` (covered
  in the previous cell) — note `state = await handle_refund(state)` **reassigns**
  `state` to whatever `handle_refund` returns, since that function may have added
  refund-specific fields or changed `status`.
- The final `else` branch handles the third possible intent, `"general"` — no LLM call
  at all here; it's just a static, canned response, since general inquiries in this demo
  aren't automated any further (a natural place a "Billing Agent," suggested in the
  student challenges, could plug in).
- `return state` — the fully updated state is handed back to the caller.

In [ ]:
async def route_customer(state: SupportState) -> SupportState:

    if state.intent == "technical":
        technical = await run_technical_support(
            state.customer_message
        )

        if technical.resolved:
            state.status = "RESOLVED"
            state.response = technical.answer
            state.technical_solution = technical.answer
        else:
            state.status = "ESCALATED"
            state.escalated = True
            state.escalation_reason = technical.escalation_reason
            state.response = (
                "I could not safely resolve this issue automatically. "
                "Your request has been escalated to a support specialist."
            )

    elif state.intent == "refund":
        state = await handle_refund(state)

    else:
        state.status = "GENERAL_SUPPORT"
        state.response = (
            "A general support representative will review your request."
        )

    return state

## 10. Conversation API simulation

The entire call chain is async. Do not use `asyncio.run()` inside Colab/Jupyter.

### 🔎 Line-by-line — Conversation API simulation

```python
async def support_request(
    customer_id: str,
    message: str,
    existing_state: Optional[SupportState] = None,
) -> SupportState:

    if existing_state is None:
        workflow_state = SupportState(
            customer_id=customer_id,
            customer_message=message,
        )

        triage = await run_triage(message)

        workflow_state.intent = triage.intent
        workflow_state.priority = triage.priority
        workflow_state.triage_reason = triage.reason
        workflow_state.status = "TRIAGED"
```

- `support_request` is the **single public entry point** simulating what an API
  endpoint would look like in a real system (e.g., `POST /support/message`). It's the
  function every "Scenario" cell later in the notebook actually calls.
- `existing_state: Optional[SupportState] = None` — this parameter is what makes
  pause/resume possible: if it's `None`, this is treated as a **brand-new**
  conversation; if a previous `SupportState` is passed in, this is treated as a
  **continuation** of an existing one.
- **New conversation branch** (`if existing_state is None:`):
  1. Create a fresh `SupportState` with just the `customer_id` and the first `message`.
  2. Run the Triage Agent (`await run_triage(message)`) to classify intent/priority.
  3. Copy the triage results into the new state, and mark `status = "TRIAGED"`.

```python
    else:
        workflow_state = existing_state.model_copy(deep=True)
        workflow_state.customer_message = message
        workflow_state.response = None

    return await route_customer(workflow_state)
```

- **Resuming conversation branch** (`else:`):
  - `existing_state.model_copy(deep=True)` — Pydantic's built-in method to create a
    full, independent **copy** of the existing state (a *deep* copy means nested
    objects like `missing_fields` are copied too, not just referenced) — this avoids
    accidentally mutating the caller's original `existing_state` object in place, which
    could cause confusing bugs if that same object is reused or inspected elsewhere.
  - `workflow_state.customer_message = message` — overwrites just the message field
    with whatever the customer just said in *this* turn.
  - `workflow_state.response = None` — clears out the *previous* turn's response, since
    a new one is about to be generated; this avoids accidentally displaying stale
    output if something goes wrong downstream.
  - Notice: on resume, **triage does not run again** — the previously-determined
    `intent` is preserved from before, which is exactly why the conversation can
    correctly route straight back into `handle_refund` and pick up gathering the
    missing information, rather than being re-classified from scratch.
- Either way, execution ends by calling `await route_customer(workflow_state)`, which
  dispatches to the correct handler based on `intent`, exactly as covered in the
  previous cell.

```python
def print_result(state: SupportState):
    print("\n" + "=" * 70)
    print("CUSTOMER SUPPORT RESULT")
    print("=" * 70)
    print(f"Status   : {state.status}")
    print(f"Intent   : {state.intent}")
    print(f"Priority : {state.priority}")
    print(f"Response : {state.response}")
    print(f"Missing  : {state.missing_fields}")
    print(f"Order ID : {state.refund_order_id}")
    print("=" * 70)
```

- A plain (non-async, no LLM involved) helper function used by every scenario cell
  below to neatly print out the important fields of a `SupportState` after each run.
- `"=" * 70` repeats the `=` character 70 times to build a visual separator line — a
  simple formatting trick worth knowing.
- Each `print(f"... : {state.xxx}")` line labels and displays one field from the final
  state, giving a quick, consistent summary view regardless of which scenario was run.

In [ ]:
async def support_request(
    customer_id: str,
    message: str,
    existing_state: Optional[SupportState] = None,
) -> SupportState:

    if existing_state is None:
        workflow_state = SupportState(
            customer_id=customer_id,
            customer_message=message,
        )

        triage = await run_triage(message)

        workflow_state.intent = triage.intent
        workflow_state.priority = triage.priority
        workflow_state.triage_reason = triage.reason
        workflow_state.status = "TRIAGED"

    else:
        workflow_state = existing_state.model_copy(deep=True)
        workflow_state.customer_message = message
        workflow_state.response = None

    return await route_customer(workflow_state)


def print_result(state: SupportState):
    print("\n" + "=" * 70)
    print("CUSTOMER SUPPORT RESULT")
    print("=" * 70)
    print(f"Status   : {state.status}")
    print(f"Intent   : {state.intent}")
    print(f"Priority : {state.priority}")
    print(f"Response : {state.response}")
    print(f"Missing  : {state.missing_fields}")
    print(f"Order ID : {state.refund_order_id}")
    print("=" * 70)

## 10.5. Pre-flight deterministic business-rule test

This test runs without an LLM and verifies that the application-layer refund functions work before the agent workflow is started.

### 🔎 Line-by-line — Pre-flight deterministic business-rule test

```python
assert _validate_refund("ORD-1001", "CUST-1001").startswith("ELIGIBLE:")
assert _validate_refund("ORD-1002", "CUST-1001").startswith("NOT_ELIGIBLE:")
print("Pre-flight business-rule test passed.")
```

- `assert <condition>` — a Python statement that does nothing if `<condition>` is
  `True`, but immediately raises an `AssertionError` (stopping the notebook) if it's
  `False`. This is a lightweight way to write inline tests directly in a script/notebook
  without a full testing framework.
- **This cell deliberately calls no LLM at all** — it's testing the *deterministic*
  `_validate_refund` function directly, in isolation, with **no involvement from Gemini
  or CrewAI whatsoever**. This is an important lesson: the parts of your system that
  don't need an LLM (like hard business rules) should be tested the same way you'd test
  any normal Python function — fast, free, and 100% reproducible, with no API calls or
  network flakiness involved.
- First assertion: `ORD-1001` belongs to `CUST-1001` and has `payment_status: "failed"`
  (an eligible status per `REFUND_POLICY`), so validation should return something
  starting with `"ELIGIBLE:"`.
- Second assertion: `ORD-1002` actually belongs to `CUST-1002`, not `CUST-1001` — so
  calling it with `CUST-1001` should fail the customer-match check and return something
  starting with `"NOT_ELIGIBLE:"`.
- If both assertions pass silently, the confirmation message prints, giving confidence
  that the core financial logic is correct **before** spending any API credits running
  the full LLM-driven scenarios below.

In [ ]:
assert _validate_refund("ORD-1001", "CUST-1001").startswith("ELIGIBLE:")
assert _validate_refund("ORD-1002", "CUST-1001").startswith("NOT_ELIGIBLE:")
print("Pre-flight business-rule test passed.")

## 11. Scenario A — refund request with missing order ID

Expected:

```text
Triage → REFUND
          ↓
Refund Agent
          ↓
Missing order_id
          ↓
WAITING_FOR_CUSTOMER
```

### 🔎 Line-by-line — Scenario A: refund request with a missing order ID

```python
refund_state = await support_request(
    "CUST-1001",
    "I was charged for an order but the payment failed. I want my money back.",
)

print_result(refund_state)
```

- This is the **first real, full run of the whole pipeline**, calling the
  `support_request` entry point directly at notebook top-level with `await` (this only
  works because Colab/Jupyter's notebook cells already run inside an active event loop
  — outside a notebook, you'd typically need `asyncio.run(...)` around an `async def
  main()` instead).
- `existing_state` is **not** passed, so this is treated as a brand-new conversation:
  triage will run, classify this as `intent="refund"`, and route into `handle_refund`.
- The message deliberately does **not** mention an order ID — only that a payment
  failed and the customer wants their money back. Because `handle_refund` calls
  `extract_refund_details`, and the LLM won't be able to find an `order_id` anywhere in
  this message, `missing_fields` should come back non-empty.
- Expected result (per the markdown diagram above this cell): `status` should end up as
  `"WAITING_FOR_CUSTOMER"`, and `response` should contain a clarification question
  asking for the order ID.
- `refund_state` is saved as a Python variable specifically so it can be **passed back
  in** to the next cell — this is the notebook literally demonstrating pause/resume in
  action across two separate cells.

In [ ]:
refund_state = await support_request(
    "CUST-1001",
    "I was charged for an order but the payment failed. I want my money back.",
)

print_result(refund_state)

## 12. Customer supplies the missing information

The same state is passed back into the workflow.

### 🔎 Line-by-line — Customer supplies the missing information

```python
refund_state = await support_request(
    "CUST-1001",
    "The order ID is ORD-1001 and the payment failed.",
    existing_state=refund_state,
)

print_result(refund_state)
```

- This is the **resume step**: the same `refund_state` variable from the previous
  cell (currently sitting in `"WAITING_FOR_CUSTOMER"` status) is passed in via
  `existing_state=refund_state`.
- Because `existing_state` is **not** `None` this time, `support_request` takes the
  `else` branch discussed earlier: it deep-copies the existing state, replaces just the
  `customer_message` with this new reply, clears the old `response`, and — crucially —
  **skips re-running triage**, going straight to `route_customer`, which routes back
  into `handle_refund` since `intent` is still `"refund"` from before.
- Inside `handle_refund`, `extract_refund_details` runs again — this time with the new
  message *and* the previous state (which the earlier cell's docstring described as
  "Existing order ID," etc.) — so the LLM can combine the newly mentioned
  `"ORD-1001"` with anything already known.
- Since `order_id` and `reason` (payment failed) are now both available,
  `missing_fields` should come back empty this time, allowing `handle_refund` to
  proceed past the pause point into actual validation/processing — reusing the exact
  same `_validate_refund`/`_process_refund` deterministic functions covered earlier.
- The variable is reassigned to `refund_state` again (overwriting the old, paused
  version) — the notebook prints the final, resolved result via `print_result`.

In [ ]:
refund_state = await support_request(
    "CUST-1001",
    "The order ID is ORD-1001 and the payment failed.",
    existing_state=refund_state,
)

print_result(refund_state)

## 13. Scenario B — complete refund request

### 🔎 Line-by-line — Scenario B: complete refund request in one message

```python
complete_refund_state = await support_request(
    "CUST-1002",
    "Please refund order ORD-1002 because I received the wrong product.",
)

print_result(complete_refund_state)
```

- A **contrasting scenario** to Scenario A: this time the customer's very first message
  already contains everything needed — an order ID (`ORD-1002`) and a reason ("wrong
  product") — so `extract_refund_details` should return an empty `missing_fields` list
  on the very first pass, and `handle_refund` should proceed straight through to
  validation and processing without ever pausing for `"WAITING_FOR_CUSTOMER"`.
- Recall `ORD-1002` belongs to `CUST-1002` with `payment_status: "captured"` (an
  eligible status) — so this should end in a **successful** refund
  (`status = "REFUND_COMPLETED"`), demonstrating the "happy path" end-to-end in a single
  call.
- A fresh variable name (`complete_refund_state`, not reusing `refund_state`) is used
  deliberately to keep each scenario's final state independently inspectable, rather
  than overwriting earlier results.

In [ ]:
complete_refund_state = await support_request(
    "CUST-1002",
    "Please refund order ORD-1002 because I received the wrong product.",
)

print_result(complete_refund_state)

## 14. Scenario C — technical support

### 🔎 Line-by-line — Scenario C: technical support

```python
technical_state = await support_request(
    "CUST-1001",
    "My login is not working and I cannot sign in to the application.",
)

print_result(technical_state)
```

- Switches to demonstrating the **other** major intent path: this message describes a
  login problem, which the Triage Agent should classify as `intent="technical"`, routing
  into `run_technical_support` via `route_customer`.
- Recall the `KB` dictionary defined earlier has a `"login"` entry with concrete
  guidance ("Verify credentials, reset the password, and check MFA..."), so the
  Technical Agent's `search_knowledge_base` tool call should find a relevant match, and
  the agent should be able to confidently return `resolved=True` with that guidance as
  the `answer` — resulting in `status = "RESOLVED"`.

In [ ]:
technical_state = await support_request(
    "CUST-1001",
    "My login is not working and I cannot sign in to the application.",
)

print_result(technical_state)

## 15. Scenario D — technical escalation

### 🔎 Line-by-line — Scenario D: technical escalation

```python
escalation_state = await support_request(
    "CUST-1001",
    "The production API returns a strange error every few minutes and I have no logs.",
)

print_result(escalation_state)
```

- This message is deliberately **vague and under-specified** on purpose: an
  intermittent API error with *no logs provided*, and no matching topic in the `KB`
  dictionary (which only covers `"payment failed"`, `"login"`, and `"app crash"`).
- This is designed to trigger the **honest-failure path** built into
  `run_technical_support`'s task instructions: *"If evidence is insufficient, set
  resolved=false and explain why escalation is needed."* Since the knowledge base won't
  return a confident, applicable answer, the Technical Agent should return
  `resolved=False` with an `escalation_reason` explaining that more information (like
  logs) is needed.
- In `route_customer`, this causes `status = "ESCALATED"`, `escalated = True`, and a
  reassuring canned message is shown to the customer rather than a guessed-at technical
  fix — demonstrating that this system is explicitly designed to **not** hallucinate
  confident-sounding but unsupported technical advice.

In [ ]:
escalation_state = await support_request(
    "CUST-1001",
    "The production API returns a strange error every few minutes and I have no logs.",
)

print_result(escalation_state)

## 16. Scenario E — customer/order mismatch

This tests the deterministic authorization boundary.

The LLM cannot override the `customer_id ↔ order_id` validation.

### 🔎 Line-by-line — Scenario E: customer/order mismatch (authorization boundary)

```python
invalid_refund_state = await support_request(
    "CUST-1001",
    "Please refund order ORD-1002. I received the wrong product.",
)

print_result(invalid_refund_state)
```

- The critical detail here: this request comes from **`CUST-1001`**, but
  **`ORD-1002`** actually belongs to `CUST-1002` (see the `ORDERS` dictionary). This
  message otherwise looks just as complete and legitimate as Scenario B's — a plausible
  order ID and a clear reason — which is exactly the point.
- This scenario specifically tests whether the **LLM could be tricked** (by a
  legitimate-sounding request, or by malicious intent) into approving a refund for
  someone else's order. Because `handle_refund` calls the deterministic
  `_validate_refund` function — which independently re-checks
  `order["customer_id"] != customer_id` regardless of what the LLM extracted or
  "believed" — this request should be rejected with
  `"NOT_ELIGIBLE: Customer does not match the order."`, regardless of how the request
  was phrased.
- This directly demonstrates the notebook's core enterprise-safety principle
  (also spelled out explicitly in the markdown above this cell): *"The LLM cannot
  override the `customer_id ↔ order_id` validation."* The LLM's role is limited to
  extracting/understanding intent; the actual authorization decision is enforced by
  plain, unbypassable Python logic.

In [ ]:
invalid_refund_state = await support_request(
    "CUST-1001",
    "Please refund order ORD-1002. I received the wrong product.",
)

print_result(invalid_refund_state)

## 17. Enterprise state machine

```text
NEW
 ↓
TRIAGED
 ├───────────────┐
 ↓               ↓
TECHNICAL       REFUND
 ↓               ↓
KB SEARCH       VALIDATE
 ↓               ↓
RESOLVED?      COMPLETE?
 /    \          /    \
YES    NO       NO     YES
 ↓      ↓       ↓       ↓
REPLY  ESCALATE ASK   PROCESS
                ↓
        WAITING_FOR_CUSTOMER
                ↓
         CUSTOMER RESPONSE
                ↓
              RESUME
```

`WAITING_FOR_CUSTOMER` is a real workflow state. The application keeps the state and resumes it when the customer responds.

## 18. Production architecture

```text
Customer
   ↓
API Gateway / FastAPI
   ↓
Conversation Service
   ↓
Persistent Workflow State
   ↓
Triage Agent (Gemini)
   ↓
Deterministic Router
   ├── Technical Support Agent → Knowledge Base → Resolve / Escalate
   └── Refund Agent → Order Service → Validate
                              ↓
                         Missing data?
                          /        \
                        YES         NO
                         ↓           ↓
                      Ask User    Process
                         ↓
                  Persist State
                         ↓
                  Customer Reply
                         ↓
                       Resume
```

For real financial systems, authentication, authorization, idempotency, audit logging, transaction controls and human approval thresholds should be implemented outside the LLM.

## 19. CrewAI vs LangGraph

### CrewAI view

```text
Crew
 ├── Triage Agent
 ├── Technical Agent
 └── Refund Agent
      ↓
Role + Goal + Backstory + Task
```

### LangGraph view

```text
StateGraph
 ├── triage
 ├── route
 ├── technical
 ├── refund
 ├── ask_customer
 ├── resume
 ├── process_refund
 └── escalation
```

**CrewAI:** specialized agents, roles, tasks, crews and delegation.

**LangGraph:** explicit state, nodes, edges, conditional routing, persistence and workflow control.

A hybrid architecture is also possible: LangGraph can own the global stateful workflow while a CrewAI crew performs a specialist sub-workflow.

## 20. Advanced student challenges

1. Add a Billing Agent.
2. Require human approval for refunds above ₹10,000.
3. Allow at most two clarification rounds before escalation.
4. Persist `SupportState` in Redis.
5. Replace the mock KB with a vector database/RAG pipeline.
6. Add an audit trail for every agent/task/tool/decision.
7. Add retry logic for refund-service failures.
8. Add authentication and customer ownership checks.
9. Rebuild the state machine using LangGraph.
10. Build a hybrid LangGraph + CrewAI architecture.

## 21. Colab/Jupyter troubleshooting

### Error 1 — running synchronous CrewAI inside an event loop

```text
RuntimeError:
Agent execution was invoked synchronously from within a running event loop.
```

Use:

```python
result = await crew.kickoff_async()
```

This notebook uses async execution throughout.

### Error 2 — calling a CrewAI Tool like a normal Python function

```text
TypeError: 'Tool' object is not callable
```

`@tool` converts a function into a CrewAI `Tool` object.

The notebook therefore separates:

- `validate_refund` → CrewAI Tool for the agent
- `_validate_refund` → plain application function
- `process_refund` → CrewAI Tool for the agent
- `_process_refund` → plain application function

The application uses the underscore-prefixed functions for deterministic business logic.

### Important

Do not use `asyncio.run()` inside a normal Colab/Jupyter cell when an event loop is already running.

## 22. Final takeaway

```text
CrewAI
  ↓
Agents + Roles + Goals + Backstories
  ↓
Tasks + Tools + Crews
  ↓
Specialized collaboration

Human-in-the-loop
  ↓
Missing information
  ↓
Pause
  ↓
Persist state
  ↓
Customer response
  ↓
Resume

Enterprise safety
  ↓
LLM extracts/reasons
  ↓
Deterministic validation
  ↓
Authorized business operation
```

**Critical implementation rule for this notebook:** use `await crew.kickoff_async()` throughout the async call chain.